# Inference Attack Module

This notebook simulates a **passive honest-but-curious server** that:
1. Observes model updates (gradients) **or** shared representations
2. Trains an attack classifier to predict subject identity
3. Measures **Attack Success Rate (ASR)** = accuracy of identity prediction

### Two Attack Surfaces
| Surface | Description |
|---|---|
| **A. Feature-space** | Attacker has access to intermediate features |
| **B. Gradient-space** | Attacker observes gradient updates each round |

Both use a simple MLP attacker trained with `sklearn` or `PyTorch`.

## 1. Imports & Device Setup

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from typing import Dict, List, Tuple, Optional

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cpu


## 2. Feature Extraction

### 2a. Single Model Feature Extraction

In [2]:
@torch.no_grad()
def extract_features(
    model: nn.Module,
    loader: DataLoader,
    noise_std: float = 0.0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Pass data through model, collect intermediate features.

    Args:
        model:     Trained nn.Module with an `extract_features` method.
        loader:    DataLoader yielding (X, y_activity, y_subject) batches.
        noise_std: Optional Gaussian noise std added to inputs (privacy simulation).

    Returns:
        features   (N, D)  - intermediate feature vectors
        y_activity (N,)    - activity labels
        y_subject  (N,)    - subject identity labels
    """
    model.eval().to(DEVICE)
    feats_list, acts_list, subjs_list = [], [], []

    for X, y_act, y_subj in loader:
        X = X.to(DEVICE)
        if noise_std > 0:
            X = X + torch.randn_like(X) * noise_std
        feats = model.extract_features(X).cpu().numpy()
        feats_list.append(feats)
        acts_list.append(y_act.numpy())
        subjs_list.append(y_subj.numpy())

    model.cpu()
    return (
        np.concatenate(feats_list),
        np.concatenate(acts_list),
        np.concatenate(subjs_list),
    )

### 2b. Ensemble Feature Extraction

In [3]:
@torch.no_grad()
def extract_ensemble_features(
    models: List[nn.Module],
    loader: DataLoader,
    noise_std: float = 0.0,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Concatenated features from all ensemble models.

    Args:
        models:    List of nn.Module models, each with `extract_features`.
        loader:    DataLoader yielding (X, y_activity, y_subject) batches.
        noise_std: Optional input noise std.

    Returns:
        features   (N, D*len(models)) - concatenated feature vectors
        y_activity (N,)
        y_subject  (N,)
    """
    for m in models:
        m.eval().to(DEVICE)

    all_feats, all_acts, all_subjs = [], [], []

    for X, y_act, y_subj in loader:
        X = X.to(DEVICE)
        if noise_std > 0:
            X = X + torch.randn_like(X) * noise_std
        feats = torch.cat([m.extract_features(X).cpu() for m in models], dim=1).numpy()
        all_feats.append(feats)
        all_acts.append(y_act.numpy())
        all_subjs.append(y_subj.numpy())

    for m in models:
        m.cpu()

    return (
        np.concatenate(all_feats),
        np.concatenate(all_acts),
        np.concatenate(all_subjs),
    )

## 3. Gradient Extraction

Simulates the server observing gradient uploads from clients.

In [4]:
def extract_gradients(
    model: nn.Module,
    loader: DataLoader,
    n_batches: int = 5,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Compute per-sample gradient norms from the first layer as attack signal.

    Args:
        model:     nn.Module to compute gradients from.
        loader:    DataLoader yielding (X, y_activity, y_subject) batches.
        n_batches: Number of batches to process (limits compute).

    Returns:
        gradient_features (N, P) - flattened gradient vectors per sample
        y_subject         (N,)   - subject identity labels
    """
    model.train().to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    grad_feats, subj_list = [], []

    for i, (X, y_act, y_subj) in enumerate(loader):
        if i >= n_batches:
            break
        X, y_act = X.to(DEVICE), y_act.to(DEVICE)

        # Compute per-sample gradients (approximate)
        batch_grads = []
        for xi, yi in zip(X, y_act):
            model.zero_grad()
            out  = model(xi.unsqueeze(0))
            loss = criterion(out, yi.unsqueeze(0))
            loss.backward()
            # Flatten all gradients into one vector
            g = torch.cat([p.grad.view(-1) for p in model.parameters()
                           if p.grad is not None])
            batch_grads.append(g.detach().cpu().numpy())

        grad_feats.append(np.stack(batch_grads))
        subj_list.append(y_subj.numpy())

    model.cpu()
    return np.concatenate(grad_feats), np.concatenate(subj_list)

## 4. Attack Classifiers

### 4a. Feature-Space Attack

Trains a classifier on model features to predict subject identity.

In [5]:
class FeatureSpaceAttack:
    """
    Trains a classifier on model features to predict subject identity.
    Uses cross-validation for a fair evaluation.

    Args:
        clf_type: 'rf' (Random Forest) | 'lr' (Logistic Regression)
                  RF is stronger; LR is faster with probability calibration.
    """

    def __init__(self, clf_type: str = "rf"):
        self.clf_type = clf_type
        self.scaler   = StandardScaler()
        self.clf      = (
            RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
            if clf_type == "rf"
            else LogisticRegression(
                max_iter=500, C=1.0, solver="saga",
                multi_class="multinomial", n_jobs=-1
            )
        )
        self.fitted = False

    def fit(self, features: np.ndarray, y_subj: np.ndarray) -> "FeatureSpaceAttack":
        features = self.scaler.fit_transform(features)
        self.clf.fit(features, y_subj)
        self.fitted = True
        return self

    def attack_success_rate(
        self,
        features: np.ndarray,
        y_subj:   np.ndarray,
        cv:       int = 5,
    ) -> Dict:
        """
        Evaluate attack via stratified cross-validation.

        Returns dict with:
          asr      - mean cross-val accuracy (main metric)
          asr_std  - std across folds
          baseline - majority-class baseline
          report   - sklearn classification_report string
        """
        features_sc = self.scaler.fit_transform(features)

        skf    = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
        scores = cross_val_score(
            self.clf, features_sc, y_subj,
            cv=skf, scoring="accuracy", n_jobs=-1
        )

        # Fit on all data for the classification report
        self.clf.fit(features_sc, y_subj)
        preds = self.clf.predict(features_sc)

        unique, counts = np.unique(y_subj, return_counts=True)
        baseline = counts.max() / len(y_subj)

        return {
            "asr":      scores.mean(),
            "asr_std":  scores.std(),
            "baseline": baseline,
            "report":   classification_report(y_subj, preds, zero_division=0),
        }

### 4b. Gradient-Space Attack

Simulates a server watching gradient uploads. Uses PCA to handle high-dimensional gradient vectors.

In [6]:
class GradientSpaceAttack:
    """
    Trains a classifier on gradient-derived features to predict subject identity.

    More realistic: simulates a server watching gradient uploads.
    Gradient vectors are often high-dimensional, so PCA is applied first.

    Args:
        n_components: Number of PCA components to retain.
    """

    def __init__(self, n_components: int = 50):
        self.pca    = PCA(n_components=n_components)
        self.scaler = StandardScaler()
        self.clf    = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)

    def fit_attack(
        self,
        grad_feats: np.ndarray,
        y_subj:     np.ndarray,
    ) -> Dict:
        """
        Fit and evaluate the gradient-space attack.

        Returns dict with:
          asr      - mean cross-val accuracy
          asr_std  - std across folds
          baseline - majority-class baseline
        """
        grad_feats = self.scaler.fit_transform(grad_feats)

        # Clamp n_components to available dimensions
        n_comp = min(self.pca.n_components, grad_feats.shape[1], len(y_subj) - 1)
        self.pca.n_components = n_comp
        grad_feats = self.pca.fit_transform(grad_feats)

        skf    = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        scores = cross_val_score(
            self.clf, grad_feats, y_subj,
            cv=skf, scoring="accuracy", n_jobs=-1
        )
        self.clf.fit(grad_feats, y_subj)

        unique, counts = np.unique(y_subj, return_counts=True)
        baseline = counts.max() / len(y_subj)

        return {
            "asr":      scores.mean(),
            "asr_std":  scores.std(),
            "baseline": baseline,
        }

## 5. Full Attack Pipelines

### 5a. Single Model Feature-Space Attack

In [7]:
def run_feature_attack(
    model:       nn.Module,
    test_loader: DataLoader,
    noise_std:   float = 0.0,
    clf_type:    str   = "rf",
    label:       str   = "",
) -> Dict:
    """
    End-to-end feature-space attack on a single model.

    Args:
        model:       Trained model with `extract_features` method.
        test_loader: DataLoader with (X, y_activity, y_subject).
        noise_std:   Gaussian noise added to inputs (0 = no noise).
        clf_type:    'rf' or 'lr' for attack classifier.
        label:       Optional string label for printed output.

    Returns:
        dict with asr, asr_std, baseline, report
    """
    feats, y_act, y_subj = extract_features(model, test_loader, noise_std)

    attacker = FeatureSpaceAttack(clf_type=clf_type)
    result   = attacker.attack_success_rate(feats, y_subj)

    tag = f"[{label}] " if label else ""
    print(f"{tag}Attack Success Rate : {result['asr']:.3f} +/- {result['asr_std']:.3f}")
    print(f"{tag}Majority baseline   : {result['baseline']:.3f}")
    print(f"{tag}Privacy leak        : {result['asr'] - result['baseline']:.3f} above baseline")

    return result

### 5b. Ensemble Feature-Space Attack

In [8]:
def run_ensemble_attack(
    models:      List[nn.Module],
    test_loader: DataLoader,
    noise_std:   float = 0.0,
    clf_type:    str   = "rf",
    label:       str   = "",
) -> Dict:
    """
    End-to-end feature-space attack on the ensemble (concatenated features).

    Args:
        models:      List of trained models with `extract_features`.
        test_loader: DataLoader with (X, y_activity, y_subject).
        noise_std:   Gaussian noise added to inputs.
        clf_type:    'rf' or 'lr' for attack classifier.
        label:       Optional label for printed output.

    Returns:
        dict with asr, asr_std, baseline, report
    """
    feats, y_act, y_subj = extract_ensemble_features(models, test_loader, noise_std)

    attacker = FeatureSpaceAttack(clf_type=clf_type)
    result   = attacker.attack_success_rate(feats, y_subj)

    tag = f"[{label}] " if label else ""
    print(f"{tag}Ensemble ASR : {result['asr']:.3f} +/- {result['asr_std']:.3f}")
    print(f"{tag}Baseline     : {result['baseline']:.3f}")

    return result

## 6. Quick Test with Synthetic Data

Validates the full pipeline using randomly generated data and an untrained MLP.

In [9]:
class SimpleMLP(nn.Module):
    """Minimal MLP demonstrating the attack pipeline without external dependencies."""

    def __init__(self, input_dim=561, hidden_dim=128, n_classes=6):
        super().__init__()
        self.fc1  = nn.Linear(input_dim, hidden_dim)
        self.fc2  = nn.Linear(hidden_dim, n_classes)
        self.relu = nn.ReLU()

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(self.fc1(x))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.extract_features(x))

In [10]:
# Synthetic dataset: 300 samples | 561 features | 6 activities | 30 subjects
torch.manual_seed(42)
np.random.seed(42)

N, D, N_ACT, N_SUBJ = 300, 561, 6, 30

X    = torch.randn(N, D)
yact = torch.randint(0, N_ACT,  (N,))
ysub = torch.randint(0, N_SUBJ, (N,))

ds = TensorDataset(X, yact, ysub)
dl = DataLoader(ds, batch_size=32, shuffle=False)

print(f"Dataset: {N} samples | {D} features | {N_ACT} activities | {N_SUBJ} subjects")

Dataset: 300 samples | 561 features | 6 activities | 30 subjects


In [11]:
# Feature-space attack on untrained MLP
model  = SimpleMLP()
result = run_feature_attack(model, dl, label="MLP-untrained")
print("\nClassification Report (train set):")
print(result["report"])

C:\Users\PMLS\anaconda3\envs\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


[MLP-untrained] Attack Success Rate : 0.040 +/- 0.017
[MLP-untrained] Majority baseline   : 0.067
[MLP-untrained] Privacy leak        : -0.027 above baseline

Classification Report (train set):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00        10
           2       1.00      1.00      1.00         9
           3       1.00      1.00      1.00        10
           4       1.00      1.00      1.00         4
           5       1.00      1.00      1.00         6
           6       1.00      1.00      1.00        15
           7       1.00      1.00      1.00        13
           8       1.00      1.00      1.00        11
           9       1.00      1.00      1.00        12
          10       1.00      1.00      1.00         9
          11       1.00      1.00      1.00        11
          12       1.00      1.00      1.00        10
          13       1.00      1.00      1.00      

In [12]:
# Feature-space attack with input noise (privacy simulation)
print("=== With Gaussian input noise (std=0.1) ===")
result_noisy = run_feature_attack(model, dl, noise_std=0.1, label="MLP-noisy")

=== With Gaussian input noise (std=0.1) ===


C:\Users\PMLS\anaconda3\envs\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


[MLP-noisy] Attack Success Rate : 0.037 +/- 0.007
[MLP-noisy] Majority baseline   : 0.067
[MLP-noisy] Privacy leak        : -0.030 above baseline


In [13]:
# Gradient-space attack
print("=== Gradient-space attack ===")
grad_feats, y_subj_grad = extract_gradients(model, dl, n_batches=3)
print(f"Gradient feature shape: {grad_feats.shape}")

grad_attacker = GradientSpaceAttack(n_components=50)
grad_result   = grad_attacker.fit_attack(grad_feats, y_subj_grad)

print(f"Gradient ASR : {grad_result['asr']:.3f} +/- {grad_result['asr_std']:.3f}")
print(f"Baseline     : {grad_result['baseline']:.3f}")
print(f"Privacy leak : {grad_result['asr'] - grad_result['baseline']:.3f} above baseline")

=== Gradient-space attack ===
Gradient feature shape: (96, 72710)


C:\Users\PMLS\anaconda3\envs\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=3.
  warnings.warn(


Gradient ASR : 0.042 +/- 0.015
Baseline     : 0.073
Privacy leak : -0.031 above baseline


In [14]:
# Ensemble attack (3 independent models)
print("=== Ensemble feature-space attack (3 models) ===")
ensemble        = [SimpleMLP() for _ in range(3)]
ensemble_result = run_ensemble_attack(ensemble, dl, label="Ensemble")

=== Ensemble feature-space attack (3 models) ===


C:\Users\PMLS\anaconda3\envs\venv\Lib\site-packages\sklearn\model_selection\_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(


[Ensemble] Ensemble ASR : 0.047 +/- 0.029
[Ensemble] Baseline     : 0.067


## 7. Results Summary

In [15]:
print("=" * 58)
print(f"{'Attack Scenario':<32} {'ASR':>7} {'Baseline':>9} {'Leak':>7}")
print("-" * 58)

scenarios = [
    ("Feature (no noise)",  result),
    ("Feature (noise=0.1)", result_noisy),
    ("Gradient-space",      grad_result),
    ("Ensemble",            ensemble_result),
]

for name, r in scenarios:
    leak = r['asr'] - r['baseline']
    print(f"{name:<32} {r['asr']:>7.3f} {r['baseline']:>9.3f} {leak:>7.3f}")

print("=" * 58)
print("\nNote: ASR close to baseline = low privacy leak (good).")
print("      ASR >> baseline        = significant privacy leak.")

Attack Scenario                      ASR  Baseline    Leak
----------------------------------------------------------
Feature (no noise)                 0.040     0.067  -0.027
Feature (noise=0.1)                0.037     0.067  -0.030
Gradient-space                     0.042     0.073  -0.031
Ensemble                           0.047     0.067  -0.020

Note: ASR close to baseline = low privacy leak (good).
      ASR >> baseline        = significant privacy leak.
